In [ ]:
#CELL-1

import os
import torch
import torch.nn as nn
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup Complete. Using device: {device}")

# ---------------------------------------------------------
# UPDATE THIS PATH TO YOUR GOOGLE DRIVE FOLDER
# ---------------------------------------------------------
ROOT_DIR = r'D:\Programming\dataset'

In [ ]:
#CELL-2
import os
import torch
import numpy as np
import rasterio
from torch.utils.data import Dataset

class UrbanGrowthDataset(Dataset):
    def __init__(self, root_dir):
        """
        root_dir: The path to the folder containing all your 'DL_TVM-...' folders
        Example: 'D:/Programming/dataset'
        """
        self.samples = []
        self.num_classes = 9  # Dynamic World has 9 LULC classes

        # 1. SCAN THE ROOT DIRECTORY
        # We look for folders that start with "DL_TVM-inputs-"
        if not os.path.exists(root_dir):
             print(f"❌ Error: Root path not found: {root_dir}")
             return

        all_folders = os.listdir(root_dir)
        
        # Filter for input folders only
        input_folders = [f for f in all_folders if f.startswith("DL_TVM-inputs-")]
        
        print(f"📂 Found {len(input_folders)} input folders.")

        for inp_folder_name in input_folders:
            # Construct the full path to the input folder
            input_folder_path = os.path.join(root_dir, inp_folder_name)
            
            # 2. FIND THE MATCHING LABEL FOLDER
            # Logic: Replace 'inputs' with 'labels' in the folder name
            lbl_folder_name = inp_folder_name.replace("-inputs-", "-labels-")
            label_folder_path = os.path.join(root_dir, lbl_folder_name)

            if not os.path.exists(label_folder_path):
                print(f"⚠️ Warning: Matching label folder not found for {inp_folder_name}")
                continue

            # 3. MATCH FILES INSIDE THE FOLDERS
            # We assume filenames inside are like 'input_y2016_2020_p0.tif'
            for file in os.listdir(input_folder_path):
                if file.endswith(".tif") and file.startswith("input"):
                    input_path = os.path.join(input_folder_path, file)
                    
                    # Logic: Replace 'input' with 'label' in the filename
                    # e.g., input_y2016_2020_p0.tif -> label_y2016_2020_p0.tif
                    label_name = file.replace("input", "label")
                    label_path = os.path.join(label_folder_path, label_name)

                    if os.path.exists(label_path):
                        self.samples.append((input_path, label_path))
        
        print(f"✅ Dataset Loaded: {len(self.samples)} valid pairs found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        input_path, label_path = self.samples[idx]

        # ---------------------------------------------------------
        # A. LOAD DATA
        # ---------------------------------------------------------
        with rasterio.open(input_path) as src:
            x = src.read().astype(np.float32) # Shape: (2, H, W)
        
        with rasterio.open(label_path) as src:
            y = src.read(1).astype(np.float32) # Shape: (H, W)

        # ---------------------------------------------------------
        # B. FIX NANS (The Mask Fix)
        # ---------------------------------------------------------
        x = np.nan_to_num(x, nan=0.0)
        y = np.nan_to_num(y, nan=0.0)

        # ---------------------------------------------------------
        # C. CROP TO 256x256 (The "Crash" Fix)
        # ---------------------------------------------------------
        if x.shape[1] > 256: x = x[:, :256, :]
        if x.shape[2] > 256: x = x[:, :, :256]
        if y.shape[0] > 256: y = y[:256, :]
        if y.shape[1] > 256: y = y[:, :256]

        # ---------------------------------------------------------
        # D. FEATURE ENGINEERING (The Accuracy Fix)
        # ---------------------------------------------------------
        # Band 0: LULC (Integer) | Band 1: Distance (Float)
        lulc = x[0, :, :].astype(np.int64)
        dist = x[1, :, :].astype(np.float32)

        # Normalize Distance (0 to 1)
        dist = np.clip(dist / 5000.0, 0, 1)

        # One-Hot Encode LULC -> Turns 1 band into 9 bands
        lulc_onehot = np.eye(self.num_classes)[lulc]

        # Combine: 9 LULC bands + 1 Distance band = 10 Input Channels
        combined_input = np.concatenate([lulc_onehot, np.expand_dims(dist, axis=-1)], axis=-1)

        # ---------------------------------------------------------
        # E. CONVERT TO TENSOR
        # ---------------------------------------------------------
        x_tensor = torch.from_numpy(combined_input).permute(2, 0, 1).float()
        y_tensor = torch.from_numpy(y).unsqueeze(0).float()

        return x_tensor, y_tensor

In [ ]:
#CELL-3
class UNet(nn.Module):
    def __init__(self, in_channels=10, out_channels=1):
        super(UNet, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            )

        # Encoder
        self.enc1 = conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = conv_block(256, 512)

        # Decoder
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256) # 256+256
        
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        
        b = self.bottleneck(self.pool3(e3))
        
        d3 = self.up3(b)
        # Resize to match if needed (safe for odd dims)
        if d3.shape != e3.shape: d3 = torch.nn.functional.interpolate(d3, size=e3.shape[2:])
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        if d2.shape != e2.shape: d2 = torch.nn.functional.interpolate(d2, size=e2.shape[2:])
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        if d1.shape != e1.shape: d1 = torch.nn.functional.interpolate(d1, size=e1.shape[2:])
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final(d1)

In [ ]:
#CELL-4
# 1. Initialize Dataset & Loader
dataset = UrbanGrowthDataset(ROOT_DIR)

# Split Train/Val (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

# 2. Initialize Model & Loss
model = UNet(in_channels=10, out_channels=1).to(device)

# WEIGHTED LOSS: Vital for sparse growth (15x weight on "Growth" pixels)
pos_weight = torch.tensor([15.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 3. Training Loop
num_epochs = 20
print("🚀 Starting Training...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

# Save Model
torch.save(model.state_dict(), 'tvm_growth_model.pth')
print("✅ Model Saved!")

In [1]:
#CELL-5
model.eval()
with torch.no_grad():
    # Get a batch from validation
    inputs, labels = next(iter(val_loader))
    inputs = inputs.to(device)
    
    # Predict
    preds = model(inputs)
    probs = torch.sigmoid(preds)

# Move to CPU for plotting
inputs = inputs.cpu().numpy()
labels = labels.cpu().numpy()
probs = probs.cpu().numpy()

# Pick the first image
idx = 0

# Extract Data for Visualization
# Inputs shape is (10, 256, 256). 
# Channel 6 is the "Urban" One-Hot channel (Class 6).
current_urban = inputs[idx, 6, :, :] 

# Prediction
predicted_growth = probs[idx, 0, :, :]
actual_growth = labels[idx, 0, :, :]

# CREATE THE "FUTURE EXTENT" MAP (Combine Old + New)
future_extent = np.maximum(current_urban, predicted_growth)

# Plot
plt.figure(figsize=(16, 4))
plt.subplot(1, 4, 1); plt.imshow(current_urban, cmap='gray'); plt.title("Current Urban (2021)")
plt.subplot(1, 4, 2); plt.imshow(actual_growth, cmap='gray'); plt.title("Actual Growth (Label)")
plt.subplot(1, 4, 3); plt.imshow(predicted_growth, cmap='magma'); plt.title("Predicted Probability")
plt.subplot(1, 4, 4); plt.imshow(future_extent, cmap='jet'); plt.title("Future Urban Extent (Goal)")
plt.show()

NameError: name 'model' is not defined

NameError: name 'UNet' is not defined

In [6]:
import os
import torch
import torch.nn as nn
import numpy as np
import rasterio
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, jaccard_score

# ---------------------------------------------------------
# 1. DEFINE THE DATASET (Needed to load the test images)
# ---------------------------------------------------------
class UrbanGrowthDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.num_classes = 9  
        if not os.path.exists(root_dir):
             print(f"❌ Error: Root path not found: {root_dir}")
             return

        all_folders = os.listdir(root_dir)
        input_folders = [f for f in all_folders if f.startswith("DL_TVM-inputs-")]
        
        for inp_folder_name in input_folders:
            input_folder_path = os.path.join(root_dir, inp_folder_name)
            lbl_folder_name = inp_folder_name.replace("-inputs-", "-labels-")
            label_folder_path = os.path.join(root_dir, lbl_folder_name)

            if not os.path.exists(label_folder_path): continue

            for file in os.listdir(input_folder_path):
                if file.endswith(".tif") and file.startswith("input"):
                    input_path = os.path.join(input_folder_path, file)
                    label_name = file.replace("input", "label")
                    label_path = os.path.join(label_folder_path, label_name)

                    if os.path.exists(label_path):
                        self.samples.append((input_path, label_path))
        
        print(f"✅ Dataset Loaded: {len(self.samples)} valid pairs found.")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        input_path, label_path = self.samples[idx]
        with rasterio.open(input_path) as src: x = src.read().astype(np.float32) 
        with rasterio.open(label_path) as src: y = src.read(1).astype(np.float32) 

        x = np.nan_to_num(x, nan=0.0)
        y = np.nan_to_num(y, nan=0.0)

        if x.shape[1] > 256: x = x[:, :256, :]
        if x.shape[2] > 256: x = x[:, :, :256]
        if y.shape[0] > 256: y = y[:256, :]
        if y.shape[1] > 256: y = y[:, :256]

        lulc = x[0, :, :].astype(np.int64)
        dist = x[1, :, :].astype(np.float32)
        dist = np.clip(dist / 5000.0, 0, 1)
        lulc_onehot = np.eye(self.num_classes)[lulc]
        combined_input = np.concatenate([lulc_onehot, np.expand_dims(dist, axis=-1)], axis=-1)

        x_tensor = torch.from_numpy(combined_input).permute(2, 0, 1).float()
        y_tensor = torch.from_numpy(y).unsqueeze(0).float()
        return x_tensor, y_tensor


# ---------------------------------------------------------
# 2. DEFINE THE UNET ARCHITECTURE
# ---------------------------------------------------------
class UNet(nn.Module):
    def __init__(self, in_channels=10, out_channels=1):
        super(UNet, self).__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            )
        self.enc1 = conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(256, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256) 
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.up3(b)
        if d3.shape != e3.shape: d3 = torch.nn.functional.interpolate(d3, size=e3.shape[2:])
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        d2 = self.up2(d3)
        if d2.shape != e2.shape: d2 = torch.nn.functional.interpolate(d2, size=e2.shape[2:])
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        if d1.shape != e1.shape: d1 = torch.nn.functional.interpolate(d1, size=e1.shape[2:])
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final(d1)

# ---------------------------------------------------------
# 3. LOAD THE MODEL AND EVALUATE
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Rebuild and Load the model weights
model = UNet(in_channels=10, out_channels=1).to(device)
try:
    model.load_state_dict(torch.load('tvm_growth_model.pth', map_location=device))
    print("✅ Model weights loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")

model.eval()

# UPDATE THIS PATH TO WHERE YOUR SMALL TEST DATASET IS NOW
TEST_DIR = r'D:\Programming\test_dataset' 

# Initialize Dataset
test_dataset = UrbanGrowthDataset(TEST_DIR)

if len(test_dataset) > 0:
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

    all_preds = []
    all_labels = []

    print("📊 Evaluating Model on Test Dataset...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    pixel_accuracy = accuracy_score(all_labels, all_preds)
    iou = jaccard_score(all_labels, all_preds, average='binary') 
    f1 = f1_score(all_labels, all_preds, average='binary')

    print("\n--- FINAL RESULTS ---")
    print(f"Pixel Accuracy : {pixel_accuracy * 100:.2f}%")
    print(f"IoU Score      : {iou:.4f}")
    print(f"F1-Score       : {f1:.4f}")
else:
    print("⚠️ Please make sure you have at least a few test images in the TEST_DIR folder to calculate accuracy!")

Using device: cuda
✅ Model weights loaded successfully!
❌ Error: Root path not found: D:\Programming\test_dataset
⚠️ Please make sure you have at least a few test images in the TEST_DIR folder to calculate accuracy!


C:\Users\Prakash\AppData\Local\Temp\ipykernel_14860\801280111.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('tvm_growth_model.pth', 

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, jaccard_score

# 1. Load the Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_channels=10, out_channels=1).to(device)

# Load your saved weights
model.load_state_dict(torch.load('tvm_growth_model.pth', map_location=device))
model.eval() # Set to evaluation mode

# 2. Load your "Mini" Test Dataset
# (Make sure to point ROOT_DIR to your new small test folder)
TEST_DIR = r'E:\LULC\lulc\test_dataset' 
test_dataset = UrbanGrowthDataset(TEST_DIR) 
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# 3. Calculate Metrics
all_preds = []
all_labels = []

print("Evaluating Model...")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Get predictions
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        
        # Convert probabilities to binary predictions (0 or 1) using a 0.5 threshold
        preds = (probs > 0.5).float()
        
        # Move to CPU and flatten arrays for scikit-learn metrics
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

# Convert to 1D numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 4. Print the Results for your Seminar!
pixel_accuracy = accuracy_score(all_labels, all_preds)
iou = jaccard_score(all_labels, all_preds) # Jaccard index is IoU
f1 = f1_score(all_labels, all_preds)

print(f"📊 Model Evaluation Results:")
print(f"Pixel Accuracy: {pixel_accuracy * 100:.2f}%")
print(f"IoU (Intersection over Union): {iou:.4f}")
print(f"F1-Score: {f1:.4f}")